# Loan Approval Synthetic Dataset Analysis and Training

## Objective
This notebook performs data understanding, preprocessing, model training, model comparison, and deployment artifact generation for the synthetic loan approval dataset.

## Dataset
`../Dataset/LoanApprovalSynthetic/loan_approval_dataset.csv`

## Deployment Goal
The final saved files from this notebook are:
- `artifacts/preprocessor.pkl`
- `artifacts/model.pkl`
- `artifacts/synthetic_model_results.csv`

These are the files used by the Flask application during prediction.

In [ ]:
from pathlib import Path

import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

ROOT = Path.cwd().resolve().parent if Path.cwd().name.lower() == "notebook" else Path.cwd().resolve()
DATASET_PATH = ROOT / "Dataset" / "LoanApprovalSynthetic" / "loan_approval_dataset.csv"
ARTIFACTS_DIR = ROOT / "artifacts"
SYNTHETIC_ARTIFACTS_DIR = ARTIFACTS_DIR / "LoanApprovalSynthetic"

ARTIFACTS_DIR.mkdir(exist_ok=True)
SYNTHETIC_ARTIFACTS_DIR.mkdir(exist_ok=True)

df = pd.read_csv(DATASET_PATH)
df.head()

In [ ]:
print("Dataset path:", DATASET_PATH)
print("Shape:", df.shape)
print("\nColumns before cleaning:\n", df.columns.tolist())
df.info()

## Data Cleaning
- Strip extra spaces from column names and text values.
- Remove duplicate rows if present.
- Drop `loan_id` because it is only an identifier and should not influence model learning.

In [ ]:
clean_df = df.copy()
clean_df.columns = clean_df.columns.str.strip()

object_columns = clean_df.select_dtypes(include=["object", "string"]).columns
for column in object_columns:
    clean_df[column] = clean_df[column].str.strip()

print("Missing values before cleaning:\n", clean_df.isnull().sum())
print("\nDuplicate rows before cleaning:", clean_df.duplicated().sum())

clean_df = clean_df.drop_duplicates().copy()
clean_df = clean_df.drop(columns=["loan_id"])

print("\nShape after cleaning:", clean_df.shape)
clean_df.head()

In [ ]:
print(clean_df["education"].value_counts())
print()
print(clean_df["self_employed"].value_counts())
print()
print(clean_df["loan_status"].value_counts())

## Feature Setup
The deployed application sends the following 11 input features, so the training pipeline must use the same schema.

In [ ]:
target_column = "loan_status"
categorical_features = ["education", "self_employed"]
numerical_features = [
    "no_of_dependents",
    "income_annum",
    "loan_amount",
    "loan_term",
    "cibil_score",
    "residential_assets_value",
    "commercial_assets_value",
    "luxury_assets_value",
    "bank_asset_value",
]

X = clean_df[categorical_features + numerical_features]
y = clean_df[target_column]

print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)

In [ ]:
num_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

cat_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(transformers=[
    ("num", num_pipeline, numerical_features),
    ("cat", cat_pipeline, categorical_features),
])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Training features shape after preprocessing:", X_train_processed.shape)
print("Testing features shape after preprocessing:", X_test_processed.shape)

## Model Training and Comparison
We train three classifiers and compare them using accuracy, precision, recall, and F1 score.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42, max_depth=8, min_samples_split=10),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        min_samples_split=4,
        min_samples_leaf=2,
    ),
}

results = []
fitted_models = {}

for name, model in models.items():
    model.fit(X_train_processed, y_train)
    predictions = model.predict(X_test_processed)
    fitted_models[name] = model
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, pos_label="Approved"),
        "Recall": recall_score(y_test, predictions, pos_label="Approved"),
        "F1 Score": f1_score(y_test, predictions, pos_label="Approved"),
    })
    print(f"\n{name} Classification Report\n")
    print(classification_report(y_test, predictions))

results_df = pd.DataFrame(results).sort_values(by=["F1 Score", "Accuracy"], ascending=False).reset_index(drop=True)
results_df

In [ ]:
best_model_name = results_df.loc[0, "Model"]
best_model = fitted_models[best_model_name]

print("Best model:", best_model_name)
print(results_df.loc[0])

## Save Deployment Artifacts
The following cell saves the preprocessing object and best model using the exact artifact names expected by the Flask app.

In [ ]:
results_path = ARTIFACTS_DIR / "synthetic_model_results.csv"
preprocessor_path = ARTIFACTS_DIR / "preprocessor.pkl"
model_path = ARTIFACTS_DIR / "model.pkl"
synthetic_model_path = SYNTHETIC_ARTIFACTS_DIR / "best_loan_model.pkl"

results_df.to_csv(results_path, index=False)
joblib.dump(preprocessor, preprocessor_path)
joblib.dump(best_model, model_path)
joblib.dump(best_model, synthetic_model_path)

print("Saved:", results_path)
print("Saved:", preprocessor_path)
print("Saved:", model_path)
print("Saved:", synthetic_model_path)

In [ ]:
sample_prediction = best_model.predict(preprocessor.transform(X_test.head(5)))
comparison_df = X_test.head(5).copy()
comparison_df["actual_status"] = y_test.head(5).values
comparison_df["predicted_status"] = sample_prediction
comparison_df

## Conclusion
- This notebook trains the model only on `Dataset/LoanApprovalSynthetic/loan_approval_dataset.csv`.
- The saved deployment artifacts are generated from the same synthetic dataset.
- After running the notebook, the Flask app can load `artifacts/preprocessor.pkl` and `artifacts/model.pkl` directly.